In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def budget_range(budget):
    if budget < 20000000:
        return "Baja"
    elif 100 <= budget < 50000000:
        return "Media"
    else:
        return "Alta"

In [0]:
budget_udf = F.udf(budget_range, StringType())

In [0]:
df_movie = spark.table(f"{catalogo}.{esquema_source}.movie")
df_actors = spark.table(f"{catalogo}.{esquema_source}.actors").withColumnRenamed("personname","person_name")

In [0]:
df_movie = df_movie.dropna(how="all")\
                        .filter((col("movieId").isNotNull()) | (col("budget")).isNotNull())

df_movie_cast = spark.table(f"{catalogo}.{esquema_source}.movie_cast").dropna(how="all")\
                    .filter((col("movieId").isNotNull()) | (col("personId")).isNotNull())

In [0]:
df_movie = df_movie.withColumn("budget_range", budget_udf("budget"))

In [0]:
df_joined = df_movie.alias("x").join(df_movie_cast.alias("y"), col("x.movieId") == col("y.movieId"), "inner")

In [0]:
df_filtered_sorted = df_joined.filter(df_movie.yearReleaseDate > 2000).orderBy("x.yearReleaseDate")

In [0]:
df_filtered_sorted = df_filtered_sorted.withColumn(
    "years_diferences", 
    F.year(F.current_date()) - F.col("yearReleaseDate")
)

In [0]:
df_aggregated = df_filtered_sorted.groupBy("yearReleaseDate", "movieStatus").agg(
    F.count(F.col('x.yearReleaseDate')).alias("num_races_2000")
)

In [0]:
df_with_budget_diff = df_filtered_sorted.withColumn(
    "budget_diff", 
    F.abs(df_filtered_sorted["budget"] - df_filtered_sorted["budget"].alias("budget")).cast(IntegerType())
)

In [0]:
df_updated = df_with_budget_diff.select("*",
                                    when(col("movieStatus").isin("Released"),
                                    when((col("budget")>-20000000) & (col("budget")<50000000), True).otherwise(False)).otherwise(False).alias("budget_flag"))

In [0]:
df_updated.join(df_actors, col("y.personId") == df_actors["person_Id"], "left") \
    .select(col("x.movieId"), "title", "budget", "yearReleaseDate", "releaseDate",
            col("durationTime").cast("string"), "characterName", col("y.personId"), "genderId",
            col("person_name").alias("personname"), "lastname") \
    .write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.movie_transformed")